# 02 - Prepare the data

Keep valid rows, save bad rows separately, and split identical profiles together. Inputs: the raw CSV and the audit from Notebook 01. Run all cells in order.

In [1]:
from pathlib import Path
import json
import platform
import sys

assert platform.system() == "Linux", "Use the WSL project kernel."
assert Path(sys.prefix).name == "mlop-pj", "Select the project environment."
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if p.name == "MLOP-PJ" and (p / "README.md").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data import prepare_data, load_splits


## 1. Check the data and model code

These small tests check parsing, duplicate groups, training-only preprocessing, and input validation. Stop if any test fails.

In [2]:
import subprocess
result = subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"],
                        cwd=ROOT, capture_output=True, text=True)
print(result.stdout + result.stderr)
(ROOT / "reports/data-tests.txt").write_text(result.stdout + result.stderr)
assert result.returncode == 0, "Fix the failed check before preparing data."


test_duplicate_groups_and_split_are_repeatable (test_workflow.DataTests.test_duplicate_groups_and_split_are_repeatable) ... ok
test_numeric_errors_are_not_repaired (test_workflow.DataTests.test_numeric_errors_are_not_repaired) ... ok
test_preparation_checks_hash_and_excludes_ids (test_workflow.DataTests.test_preparation_checks_hash_and_excludes_ids) ... ok
test_quarantine_preserves_bad_row_and_zeros (test_workflow.DataTests.test_quarantine_preserves_bad_row_and_zeros) ... ok
test_imputer_and_scaler_fit_only_training_data (test_workflow.ModelTests.test_imputer_and_scaler_fit_only_training_data) ... ok
test_predict_rejects_bad_contract_and_accepts_named_order (test_workflow.ModelTests.test_predict_rejects_bad_contract_and_accepts_named_order) ... ok
test_study_ignores_test_data_and_bundle_reload_matches (test_workflow.ModelTests.test_study_ignores_test_data_and_bundle_reload_matches) ... ok

----------------------------------------------------------------------
Ran 7 tests in 0.477s

OK


## 2. Save the split

Use 70% train, 15% validation, and the rest for test, by unique profile groups. Row shares may differ slightly. Keep zeros unchanged. No scaler or imputer is fitted here.

In [3]:
prepared = prepare_data(ROOT)
print(json.dumps({key: prepared[key] for key in ["accepted_rows", "quarantined_rows", "split_counts",
                                               "unique_profiles", "zero_policy"]}, indent=2))


{
  "accepted_rows": 5454,
  "quarantined_rows": 2,
  "split_counts": {
    "train": 3817,
    "test": 820,
    "validation": 817
  },
  "unique_profiles": 5451,
  "zero_policy": "unchanged; compare in Notebook 03"
}


## 3. Check the result

Expect 5,454 accepted rows and 2 quarantined rows from the reviewed raw version. IDs must not be model features, and identical profiles must not cross splits.

In [4]:
import pandas as pd
splits, prepared = load_splits(ROOT)
frame = pd.read_csv(ROOT / "data/processed/profiles.csv")
crossed = int((frame.groupby("profile_hash")["split"].nunique() > 1).sum())
assert crossed == 0
assert sum(len(part) for part in splits.values()) == prepared["accepted_rows"]
assert all(list(part.columns) == prepared["features"] for part in splits.values())
print("Cross-split duplicate groups:", crossed)
print("Model features:", len(prepared["features"]), "(no IDs)")
print("Ready for Notebook 03.")


Cross-split duplicate groups: 0
Model features: 24 (no IDs)
Ready for Notebook 03.


## Output

`data/processed/profiles.csv`, `data/processed/quarantine.csv`, and `reports/prepared-data.json`. Continue to 03 when all checks pass.